In [ ]:
"""
=============================================================
 NOTEBOOK 2b — Panellist Bias Detection & Correction
=============================================================
This notebook identifies and corrects systematic scoring bias
introduced by individual sensory panellists before modelling.

Why this matters:
  ICC (Intraclass Correlation Coefficient) sets the theoretical
  ceiling on any model's R². If panellists disagree systematically,
  the ceiling is low regardless of model complexity. Correcting
  bias raises this ceiling and improves label reliability.

Steps:
  1. Dataset expansion — retain individual panellist rows
     rather than averaging (exposes true within-animal noise)
  2. ANOVA — confirm panellist identity drives score variance
  3. Identify most biased panellists per component
  4. Component-level bias correction per panellist
     (Tenderness, Juiciness, Flavour corrected independently)
  5. Recompute EQ from corrected components
  6. Sanity checks — mean preserved, variance reduced

Inputs:
  - Merged panellist-level dataset (output of 01_data_merge)

Outputs:
  - Dataset with EQ_component_corrected target column
  - Per-panellist bias tables
=============================================================
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## 1. Load Data

In [ ]:
# Load the merged panellist-level dataset from notebook 01
path = 'C:/'
input_file = f"{path}ALL_LD_meat_6.0.csv"  # UPDATE PATH

df = pd.read_csv(input_file, encoding='utf-8-sig')
df.columns = df.columns.str.replace(r'\ufeff|\ufffe', '', regex=True).str.strip()

print(f"Dataset shape: {df.shape}")
print(f"Unique animals (Eartag): {df['Eartag'].nunique()}")
print(f"Unique panellists: {df['Username'].nunique()}")
print(f"Primals: {df['Primal'].unique()}")

## 2. ANOVA — Confirm Panellist Effect
Before correcting, confirm statistically that panellist identity
drives systematic score differences rather than random variation.

In [ ]:
TARGETS   = ['Tenderness', 'Flavour', 'Juicy']
MIN_OBS   = 50
BIAS_FLAG = 0.40

grader_counts  = df.groupby('Username')[TARGETS[0]].count()
active_graders = grader_counts[grader_counts >= MIN_OBS].index.tolist()
dfa            = df[df['Username'].isin(active_graders)].copy()

overall_means = {t: df[t].mean() for t in TARGETS}
overall_stds  = {t: df[t].std()  for t in TARGETS}

print(f"Total graders in dataset:        {df['Username'].nunique()}")
print(f"Active graders (>= {MIN_OBS} obs):    {len(active_graders)}")
print(f"Observations covered:            {len(dfa):,} / {len(df):,}  ({100*len(dfa)/len(df):.1f}%)")
print()
print("Overall dataset means:")
for t in TARGETS:
    print(f"  {t:<15}: {overall_means[t]:.3f}  (SD={overall_stds[t]:.3f})")

print()
print("=" * 60)
print("ONE-WAY ANOVA: Is grader effect statistically significant?")
print("=" * 60)
print("Tests whether mean scores differ significantly across graders.")
print()
for target in TARGETS:
    groups = [dfa[dfa['Username'] == g][target].dropna().values for g in active_graders]
    groups = [g for g in groups if len(g) >= 5]
    f_stat, p_val = stats.f_oneway(*groups)
    sig = '*** SIGNIFICANT' if p_val < 0.05 else 'not significant'
    print(f"  {target:<15}: F = {f_stat:>8.3f},  p = {p_val:.6f}  ->  {sig}")

## 3. Raw Mean & Deviation per Panellist

In [ ]:
PRIMALS = ['LD', 'SM', 'FT', 'GM']
primal_means = dfa.groupby('Primal')[TARGETS].mean()

grader_stats = dfa.groupby('Username')[TARGETS].agg(['mean', 'std', 'count'])
grader_stats.columns = ['_'.join(c) for c in grader_stats.columns]

print("RAW MEAN & DEVIATION PER GRADER")
for target in TARGETS:
    print(f"\nTarget: {target}  (overall mean = {overall_means[target]:.3f})")
    print(f"  {'Grader':<28} {'n':>6}  {'Mean':>7}  {'SD':>7}  {'Deviation':>10}  Flag")
    print("  " + "-" * 68)
    col_mean  = f"{target}_mean"
    col_std   = f"{target}_std"
    col_count = f"{target}_count"
    rows = []
    for grader in active_graders:
        mn  = grader_stats.loc[grader, col_mean]
        sd  = grader_stats.loc[grader, col_std]
        n   = int(grader_stats.loc[grader, col_count])
        dev = mn - overall_means[target]
        flag = '*** HIGH' if dev > BIAS_FLAG else ('*** LOW' if dev < -BIAS_FLAG else '')
        rows.append((grader, n, mn, sd, dev, flag))
    rows.sort(key=lambda x: x[4], reverse=True)
    for grader, n, mn, sd, dev, flag in rows:
        print(f"  {grader:<28} {n:>6}  {mn:>7.3f}  {sd:>7.3f}  {dev:>+10.3f}  {flag}")

## 4. Component-Level Bias Correction

In [ ]:
def apply_component_bias_correction(df: pd.DataFrame,
                                     components: list,
                                     panellist_col: str = 'Username',
                                     eq_col: str = 'EQ') -> pd.DataFrame:
    """
    Apply panellist bias correction at the component score level.
    Corrected EQ is then recomputed from corrected components.

    Parameters
    ----------
    df            : source DataFrame
    components    : list of component score columns e.g. ['Juicy','Tenderness','Flavour']
    panellist_col : panellist identifier column
    eq_col        : EQ column name (will be recomputed)
    """
    df = df.copy()
    bias_tables = {}

    print("Component-level bias correction")
    print(f"{'─'*60}")

    for comp in components:
        if comp not in df.columns:
            print(f"  ⚠️  {comp} not found — skipping")
            continue

        overall_mean   = df[comp].mean()
        panellist_bias = df.groupby(panellist_col)[comp].mean() - overall_mean
        bias_tables[comp] = panellist_bias

        df[f'{comp}_corrected'] = df.apply(
            lambda row: row[comp] - panellist_bias[row[panellist_col]], axis=1
        ).clip(1, 9)

        print(f"  {comp:<15} bias range: {panellist_bias.min():+.3f} to "
              f"{panellist_bias.max():+.3f}  "
              f"(std: {panellist_bias.std():.3f})")

    return df, bias_tables

In [ ]:
# Apply component-level bias correction
components = ['Juicy', 'Tenderness', 'Flavour']
df, bias_tables = apply_component_bias_correction(df, components)

# Recompute EQ using ABP weights from corrected components
# EQ = Tenderness (×0.4) + Juicy (×0.3) + Flavour (×0.3)
df['EQ_component_corrected'] = (
    0.4 * df['Tenderness_corrected'] +
    0.3 * df['Juicy_corrected']      +
    0.3 * df['Flavour_corrected']
).clip(1, 9)

## 5. Sanity Checks

In [ ]:
# Means should be very close — correction preserves the overall mean
print(f"Original EQ mean:              {df['EQ'].mean():.4f}")
print(f"Component-corrected EQ mean:   {df['EQ_component_corrected'].mean():.4f}")
print()

# Variance should be slightly lower — noise removed
print(f"Original EQ std:               {df['EQ'].std():.4f}")
print(f"Component-corrected EQ std:    {df['EQ_component_corrected'].std():.4f}")
print()

# Correlation — should be high (0.85+) but not perfect
print(f"Correlation original vs corrected EQ:")
print(df[['EQ', 'EQ_component_corrected']].corr().round(4))
print()

# Check how much the correction moved individual scores
df['EQ_correction_delta'] = df['EQ_component_corrected'] - df['EQ']
print(f"Correction delta (per row):")
print(df['EQ_correction_delta'].describe().round(4))

## 6. Identify Most Biased Panellists

In [ ]:
# Check whether panellist bias is consistent across components
# Consistent = global scoring style; variable = component-specific tendency
print("Panellist bias correlation across components:")
bias_df = pd.DataFrame({
    comp: bias_tables[comp] for comp in components
    if comp in bias_tables
})
print(bias_df.corr().round(3))
print()

# Most biased panellists per component
for comp in components:
    overall_mean   = df[comp].mean()
    panellist_bias = df.groupby('Username')[comp].mean() - overall_mean
    print(f"{comp} — most biased panellists:")
    print("  Most negative (consistently scores low):")
    print(panellist_bias.sort_values().head(3).round(3))
    print("  Most positive (consistently scores high):")
    print(panellist_bias.sort_values().tail(3).round(3))
    print()

## 7. Primal-Controlled Deviation
Adjusts for each grader's primal mix to isolate true scoring bias
from the effect of being assigned higher or lower quality animals.

In [ ]:
print("PRIMAL-CONTROLLED DEVIATION PER GRADER")
print("Method: weighted average of (grader mean - primal mean) across cuts.")
print()
for target in TARGETS:
    print(f"Target: {target}")
    print(f"  {'Grader':<28} {'n':>6}  {'Raw Mean':>9}  {'Adj Deviation':>14}  Flag")
    rows = []
    for grader in active_graders:
        sub = dfa[dfa['Username'] == grader]
        n   = len(sub)
        weighted_dev, total_weight = 0.0, 0
        for primal in PRIMALS:
            psub = sub[sub['Primal'] == primal][target].dropna()
            if len(psub) < 5:
                continue
            p_mean = primal_means.loc[primal, target] if primal in primal_means.index else overall_means[target]
            weighted_dev  += (psub.mean() - p_mean) * len(psub)
            total_weight  += len(psub)
        adj_dev = weighted_dev / total_weight if total_weight > 0 else float('nan')
        raw_mn  = sub[target].mean()
        flag    = 'HIGH' if adj_dev > BIAS_FLAG else ('LOW' if adj_dev < -BIAS_FLAG else '')
        rows.append((grader, n, raw_mn, adj_dev, flag))
    rows.sort(key=lambda x: x[3] if not np.isnan(x[3]) else 0, reverse=True)
    for grader, n, raw_mn, adj_dev, flag in rows:
        dev_str = f"{adj_dev:+.3f}" if not np.isnan(adj_dev) else '--'
        print(f"  {grader:<28} {n:>6}  {raw_mn:>9.3f}  {dev_str:>14}  {flag}")
    print()

## 8. Save Output

In [ ]:
output_file = f"{path}ALL_LD_meat_bias_corrected.csv"  # UPDATE PATH
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Saved: {output_file}")
print(f"Shape: {df.shape}")
print(f"New columns added: EQ_component_corrected, EQ_correction_delta,")
print(f"  Tenderness_corrected, Juicy_corrected, Flavour_corrected")